# Optional method: stylometry (the counting trick that unmasked J.K. Rowling)

A short optional starter for text projects.

In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
import os, re
import numpy as np
from collections import Counter

def load_gutenberg(path, url):
    if os.path.exists(path):
        raw = open(path, encoding="utf-8-sig").read()
    else:
        import requests
        raw = requests.get(url, timeout=30).text
    body = re.split(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", raw, flags=re.S)[-1]
    return re.split(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", body)[0]

frank = load_gutenberg("../data/texts/frankenstein.txt", "https://www.gutenberg.org/cache/epub/84/pg84.txt")
drac = load_gutenberg("../data/texts/dracula.txt", "https://www.gutenberg.org/cache/epub/345/pg345.txt")

FUNCTION_WORDS = ["the", "of", "and", "a", "to", "in", "is", "it", "that", "was", "for", "with", "as", "i"]

def style_vector(text):
    toks = re.findall(r"[a-z]+", text.lower()); n = len(toks) or 1
    c = Counter(toks)
    return np.array([c[w] / n for w in FUNCTION_WORDS])

# The Rowling test, on real books: split each novel in half. Halves of the SAME book
# should sit closer in function-word space than halves of different books.
f1, f2 = style_vector(frank[:len(frank)//2]), style_vector(frank[len(frank)//2:])
d1, d2 = style_vector(drac[:len(drac)//2]), style_vector(drac[len(drac)//2:])
dist = lambda a, b: round(float(np.linalg.norm(a - b)), 4)
print("Frankenstein 1st half vs 2nd half:", dist(f1, f2), " (same author)")
print("Dracula      1st half vs 2nd half:", dist(d1, d2), " (same author)")
print("Frankenstein vs Dracula:          ", dist(f1, d1), " (different authors)")
# Little words, no meaning, and yet the authors separate. This is the whole method that
# unmasked Robert Galbraith; scale it up with more function words and known samples.